In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import classification_report
import joblib


In [ ]:
MODELS_PATH = "/home/mariya/Data/pallas/models/dnam"
DATASETS_PATH = "/home/mariya/Data/pallas/datasets/dnam"

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "cal-and-test-dnam")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.dnam.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")
random_seed = 42
rf_model_path = os.path.join(MODELS_PATH, "rf_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")
pipeline_path = os.path.join(MODELS_PATH, "pipeline.pkl")
hyperparams_path = os.path.join(MODELS_PATH, "hyperparams.pkl")

In [9]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_test, y_test

(              cg00000236  cg00001269  cg00001793  cg00002769  cg00003298  \
 id                                                                         
 ALL_1252          0.9034      0.2219      0.0729      0.8919      0.6645   
 ALL_1281          0.8865      0.7952      0.6110      0.8681      0.0965   
 ALL_1290          0.9143      0.8244      0.2802      0.8825      0.5888   
 ALL_1362          0.9246      0.2002      0.1318      0.9447      0.4325   
 ALL_1378          0.8923      0.2011      0.1650      0.8971      0.4442   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2997      0.4988      0.9101      0.8813      0.9522      0.5876   
 TCGA-AB-2998      0.5311      0.8971      0.9313      0.9470      0.0195   
 TCGA-AB-3001      0.5815      0.7811      0.8904      0.8567      0.4746   
 TCGA-AB-3011      0.5835      0.8746      0.6678      0.8151      0.0258   
 TCGA-AB-3012      0.6158      0.8657      0.9433      0.8827      0.5057   

In [10]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test, y_test

(              cg00000236  cg00001269  cg00001793  cg00002769  cg00003298  \
 id                                                                         
 ALL_1252          0.9034      0.2219      0.0729      0.8919      0.6645   
 ALL_1281          0.8865      0.7952      0.6110      0.8681      0.0965   
 ALL_1290          0.9143      0.8244      0.2802      0.8825      0.5888   
 ALL_1362          0.9246      0.2002      0.1318      0.9447      0.4325   
 ALL_1378          0.8923      0.2011      0.1650      0.8971      0.4442   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2997      0.4988      0.9101      0.8813      0.9522      0.5876   
 TCGA-AB-2998      0.5311      0.8971      0.9313      0.9470      0.0195   
 TCGA-AB-3001      0.5815      0.7811      0.8904      0.8567      0.4746   
 TCGA-AB-3011      0.5835      0.8746      0.6678      0.8151      0.0258   
 TCGA-AB-3012      0.6158      0.8657      0.9433      0.8827      0.5057   

In [11]:
print("\ny_test class distribution:")
print(y_test.value_counts())


y_test class distribution:
class                   
AML, KMT2A-r                103
B-ALL, high hyperdiploid     98
B-ALL, ETV6::RUNX1           85
AML, RUNX1::RUNX1T1          75
AML, NPM1alt                 56
AML, CBFB::MYH11             54
Non-leukemic                 41
T-ALL                        32
AML, NUP98-r                 30
AML, CEBPAalt                23
B-ALL, PAX5alt               20
B-ALL, KMT2A-r               19
B-ALL, TCF3::PBX1            11
B-ALL, DUX4-r                10
AML, CBFA2T3::GLIS2          10
AML, DEK::NUP214              8
B-ALL, BCR::ABL1              8
AML, ETV6-r                   7
AML, APL-PML::RARA            7
B-ALL, iAMP21                 7
B-ALL, BCR::ABL1-like         6
B-ALL, ZNF384-r               6
B-ALL, ETV6::RUNX1-like       6
AML, KAT6A-r                  3
AML, PICALM::MLLT10           3
B-ALL, MEF2D-r                3
AML, RBM15::MKL1              3
AML, FUS::ERG                 2
B-ALL, PAX5 P80R              2
Name: count, dtype:

In [12]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'AML, KMT2A-r', 'AML, NPM1alt',
       'AML, KMT2A-r', 'AML, CEBPAalt', 'AML, CBFB::MYH11',
       'AML, CBFB::MYH11', 'AML, CEBPAalt', 'AML, RUNX1::RUNX1T1',
       'AML, CBFB::MYH11', 'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1',
       'AML, RUNX1::RUNX1T1', 'AML, CBFB::MYH11', 'AML, KMT2A-r',
       'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1', 'AML, CEBPAalt',
       'AML, CBFB::MYH11', 'AML, RUNX1::RUNX1T1', 'AML, CBFB::MYH11',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, PAX5alt',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'B-ALL, PAX5alt', 'B-ALL, high hyperdiploi

In [13]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_test)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test.shape}")

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Number of selected features: 18179
Shape of X_test_selected: (738, 18179)


In [14]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.857     0.923         7
     AML, CBFA2T3::GLIS2      0.889     0.800     0.842        10
        AML, CBFB::MYH11      0.963     0.963     0.963        54
           AML, CEBPAalt      1.000     0.957     0.978        23
        AML, DEK::NUP214      1.000     0.875     0.933         8
             AML, ETV6-r      1.000     0.571     0.727         7
           AML, FUS::ERG      1.000     1.000     1.000         2
            AML, KAT6A-r      1.000     0.333     0.500         3
            AML, KMT2A-r      0.768     0.932     0.842       103
            AML, NPM1alt      0.960     0.857     0.906        56
            AML, NUP98-r      0.944     0.567     0.708        30
     AML, PICALM::MLLT10      0.500     1.000     0.667         3
        AML, RBM15::MKL1      0.000     0.000     0.000         3
     AML, RUNX1::R

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [15]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Cpg Site": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 cpg sites driving classification (final model):")
print(gene_importances.head(20))


Top 20 cpg sites driving classification (final model):
         Cpg Site  Importance
4885   cg06623668    0.010781
5099   cg06891858    0.007182
11     cg00015639    0.007179
8663   cg12183594    0.006404
12834  cg18638389    0.005397
3844   cg05098566    0.005342
315    cg00424467    0.005170
8950   cg12678686    0.004987
6030   cg08166720    0.004862
465    cg00587115    0.004727
6149   cg08302325    0.004456
10857  cg15459165    0.004403
10584  cg14992144    0.004035
9207   cg13155823    0.003962
11508  cg16522484    0.003926
16642  cg25252197    0.003792
6247   cg08457158    0.003788
577    cg00760938    0.003778
16663  cg25287071    0.003761
9051   cg12868887    0.003747


In [16]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 1064 out of 18179

Bottom 20 genes by importance (final model):
         Cpg Site  Importance
15298  cg23079866         0.0
1609   cg02077100         0.0
8151   cg11411220         0.0
10947  cg15585794         0.0
14135  cg20931965         0.0
15270  cg23048215         0.0
15267  cg23045594         0.0
15260  cg23037386         0.0
9215   cg13177747         0.0
15287  cg23068712         0.0
13305  cg19523085         0.0
14142  cg20950011         0.0
12093  cg17458357         0.0
982    cg01320579         0.0
10347  cg14613228         0.0
16141  cg24417845         0.0
16131  cg24401219         0.0
16130  cg24399830         0.0
10257  cg14501281         0.0
744    cg00986133         0.0
